In [ ]:
## DataHandler (prices, returns)  -> Strategy  (signals) -> ExecutionHandler (cost, financing) -> Portfolio (position) -> PerformanceReporter (P&L, DD)

In [ ]:
## here should be two functions: updating the market data and generating signals
##class DataHandler    
##  def update_data(self, marketdata)
## and for out put it should get clean data like remove the NA or duplicate. and maybe also do simple calculation like log return , simple return

## def generate_signal(self, DataHandler)
## it takes clean data from update_data and convert to signal like -<, 0, 1.

## side effect: some of the function might be calculated in strategy. the data quality?



In [ ]:
## the configuration can include 
# financing cost
# commission fee
# time perios
# number of lag
# entry_threshold
# train_fraction


In [ ]:
# Class logging:
#    def trades(self, timestamp, symbol,nominal,price)
#    def positions(self,timestamp,symbol,position)
#    def error(self, timestamp,text)

In [ ]:
## it seems the unit-test plan is a test to check if each class works
## then the test plan can set up initial portfolio with 0 position and 1 million cash
## then test with giving and knowing signal to check if the portfolio position and cash change according

In [ ]:
## chapter 11

In [ ]:
## MarketEvent -> SignalEvent -> OrderEvent -> FillEvent -> Portofolio P&L 

In [ ]:
from collections import deque  # simple FIFO queue for events
from dataclasses import dataclass  # lightweight event containers
from pathlib import Path  # filesystem paths for macro and figure output

from pprint import pprint  # pretty printing of metric dictionaries
import numpy as np  # numerical arrays
import pandas as pd  # tabular time-series structures
import matplotlib.pyplot as plt  # plotting library for figures

@dataclass
class MarketEvent:
    time:pd.Timestamp
    price:float

@dataclass
class SignalEvent:
    time:pd.Timestamp
    signal:float

@dataclass
class OrderEvent:
    time:pd.Timestamp
    target_position:float

@dataclass
class FillEvent:
    time:pd.Timestamp
    position:float
    price:float
    
class EventQueue:
    """FIFO queue for events in the backtest."""

    def __init__(self) -> None:
        self._queue: deque=deque()

    def put(self, event: object) -> None:
        """Append a new event to the queue."""
        self._queue.append(event)

    def get(self) -> object | None:
        """Pop the next event or return None if the queue is empty."""
        if self._queue:
            return self._queue.popleft()
        return None

    def empty(self) -> bool:
        """Return True when there are no more events."""
        return not self._queue


## The eventqueue is empty waiting for the next signal from the MarketDataEvent.
## The MarketDataEvent triggers the strategy, and put a SignalEvent into the queue.
## the portfolio picks up the SinalEvent and generate and Order Event.
## FillEvent get this OrderEvent and convert into the filleevent. 

class HistoricalDataHandler:
    """Stream daily close prices as MarketEvent instances."""

    def __init__(
        self,
        symbol: str="EURUSD",
        start: str | None=None,
        end: str | None=None,
    ) -> None:
        self.symbol = symbol
        panel = load_eod_panel().astype(float)
        prices = panel[self.symbol].dropna()

        start_ts = pd.to_datetime(start) if start is not None else None
        end_ts = pd.to_datetime(end) if end is not None else None
        if start_ts is not None or end_ts is not None:
            prices = prices.loc[start_ts:end_ts]

        self.prices = prices
        self.index = self.prices.index
        self.pointer = 0

    def have_more_bars(self) -> bool:
        """True while there are bars left to stream."""
        return self.pointer < len(self.index)

    def stream_next(self, events: EventQueue) -> None:
        """Push the next MarketEvent to the event queue."""
        if not self.have_more_bars():
            return
        ts = self.index[self.pointer]
        price = float(self.prices.iloc[self.pointer])
        events.put(MarketEvent(time=ts, price=price))
        self.pointer += 1



## from MarketDataEvent to SignalEvent
class SmaCrossStrategy:
    def __init__(self,fast_window=20,slow_window=100):
        self.fast_window = fast_window
        self.slow_window = slow_window
        self.prices = []
        self.times = []
        self.current_signal = 0.0

    def on_market_event(self,event,events):
        self.times.append(event.time)
        self.prices.append(event.price)
        series = pd.Series(self.prices, index=self.times)

        fast = series.rolling(self.fast_window).mean()
        slow = series.rolling(self.slow_window).mean()

        if len(series) < self.slow_window:
            return # not enought data yet
        if  fast.iloc[-1] > slow.iloc[-1] and self.current_signal <= 0.0:
            self.current_signal = 1.0
            events.put(
                SignalEvent(time = event.time, signal = self.current_signal)
            )
        elif fast.iloc[-1] < slow.iloc[-1] and self.current_signal >= 0.0:
            self.current_signal = -1.0
            events.put(
                SignalEvent(time = event.time, signal = self.current_signal)
            )


## from Signal Event to Order Event
class NaivePortfolio:
    def __init__(self):
        self.current_position = 0.0
    
    def on_signal_event(self, event:SignalEvent, events:EventQueue,)-> None:
        
        events.put(
            OrderEvent(time=event.time, target_position=event.signal)
        )

## from OrderEvent to Fillevent:
class SimulatedExecutionHandler:
    """Turn orders into immediate fills at the closing price."""

    def __init__(self) -> None:
        self.last_price: float | None=None

    def on_market_event(self, event: MarketEvent) -> None:
        """Record the most recent price for use in fills."""
        self.last_price = event.price

    def on_order(
        self,
        event: OrderEvent,
        events: EventQueue,
    ) -> None:
        """Generate a fill at the latest known price."""
        if self.last_price is None:
            return
        events.put(
            FillEvent(
                time=event.time,
                position=event.target_position,
                price=self.last_price,
            )
        )


In [ ]:

# change the OrderEvent

# erlier have

#class NaivePortfolio:
#    def __init__(self):
#        self.current_position = 0.0
    
#    def on_signal_event(self, event:SignalEvent, events:EventQueue,)-> None:
        
#        events.put(
#           OrderEvent(time=event.time, target_position=event.signal)
#        )

class VolatilityPortfolio:
    def __init__(self, target_vol=0.01):
        self.current_position = 0.0
        self.target_vol = target_vol  # target daily volatility
        self.prices = []              # need price history for vol estimate

    # to get price from MarketEvent
    def on_market_event(self, event: MarketEvent) -> None:
        # keep track of prices to compute volatility
        self.prices.append(event.price)

    # get the siganl -1, 0,1 from SignalEvent
    def on_signal_event(self, event: SignalEvent, events: EventQueue) -> None:
        if len(self.prices) < 20:
            return  # not enough history yet
        
        # estimate recent volatility
        price_series = pd.Series(self.prices)
        vol = price_series.pct_change().rolling(20).std().iloc[-1]
        vol = abs(vol)
        
        # scale position size: smaller when vol is high
        if vol > 0:
            size = self.target_vol / vol
        else:
            size = 1.0
        
        # apply signal direction with volatility-scaled size
        target_position = event.signal * size
        
        events.put(
            OrderEvent(time=event.time, target_position=target_position)
        )

# so in OrderEvent, not just takeing the signal from SignalEvent, but also calculate what size should put into the order

In [ ]:

# splippage put into the fillEvent

class SimulatedExecutionHandler:
    def __init__(self, slippage_rate=0.0001) -> None:
        self.last_price = None
        self.slippage_rate = slippage_rate  # e.g. 0.01% per trade

    # get price from MarketEvent
    def on_market_event(self, event: MarketEvent) -> None:
        self.last_price = event.price

    # get order size and position from OrderEvent
    def on_order(self, event: OrderEvent, events: EventQueue) -> None:
        if self.last_price is None:
            return
        
        # apply slippage depending on direction
        if event.target_position > 0:  # buying should adding the spread
            fill_price = self.last_price * (1 + self.slippage_rate)
        elif event.target_position < 0:  # selling should deducting the spread
            fill_price = self.last_price * (1 - self.slippage_rate)
        else:
            fill_price = self.last_price  # flat, no slippage
        
        events.put(
            FillEvent(
                time=event.time,
                position=event.target_position,
                price=fill_price  # ← slippage applied here
            )
        )

# the slippage will make the performance worse

In [ ]:
## the Vectorised method calculate all dates at once.
## event-based calculate once the MraketDataEvent triggered, event by event.

## so the vetorised method has higher risk of look ahead
